# Week 3 Hands-on: Ship a scam-text filter

**7DATA001W Machine Learning in Practice** · 1 hour 15 minutes · work in pairs

### Your ticket

You are a junior data scientist at a UK mobile network. In the UK, customers can report a suspicious text by forwarding it to **7726**, and the reports land with your team. This morning's ticket:

> **DS-142 · Scam-text filter, first version** · Priority: high
>
> *"Reports are piling up: about **2 million** texts in the backlog and nobody has labelled them. I can pay annotators for **200 labels** this week. I need a first filter, and on Friday I have to tell the fraud team **what it will cost us when it gets things wrong**. Can you have something by the end of the session?"*
> — Priya, Fraud Analytics Lead

You have a sample of about 2,800 real text messages to work with. The job has three stages, the same three as today's lecture:

| Time | Stage | The question you answer for Priya |
|---|---|---|
| 0:00 – 0:05 | Setup | |
| 0:05 – 0:30 | **A · Labelling** | *Can we get labels without paying for 2 million of them?* |
| 0:30 – 0:43 | **B · Sampling** | *Can we trust the numbers we report?* |
| 0:43 – 1:10 | **C · Class imbalance** | *Which model do we ship, and what will its mistakes cost?* |
| 1:10 – 1:15 | Wrap-up | Reply to the ticket, commit, push |

**✍️ TODO** = code you write (replace each `...`). **🔮 Predict** = write your guess *before* you run the cell. **💬 Pair talk** = two minutes with your neighbour.

---
## Setup (5 min)

In the terminal:

```bash
source .venv/bin/activate
mkdir -p week-03 && cd week-03
pip install pandas scikit-learn ipykernel
```

Put this notebook and `sms_clean.csv` in `week-03/`, pick the `.venv` kernel, then run the setup cells below, up to Stage A. You do not need to edit them.

In [1]:
import re

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.utils import resample

pd.set_option("display.max_colwidth", 160)

df = pd.read_csv("sms_clean.csv")
df["y"] = (df["label"] == "spam").astype(int)   # 1 = scam/spam, 0 = ham (a genuine message)
df = df.drop(columns="label")

n_spam, n_ham = (df["y"] == 1).sum(), (df["y"] == 0).sum()
print(f"{len(df):,} messages | spam: {n_spam:,} ({n_spam / len(df):.1%}) | ham: {n_ham:,} ({n_ham / len(df):.1%})")

2,828 messages | spam: 306 (10.8%) | ham: 2,522 (89.2%)


**Split 1 · Hold back a test set.** `train_test_split` sets aside **800 messages** as `test`. We never train on these: they are only used to score models fairly at the end. The other messages go into `rest`.

`stratify=df["y"]` keeps the spam rate in `test` the same as in the full dataset, and `random_state=42` makes the split identical on every run.

In [2]:
rest, test = train_test_split(df, test_size=800, stratify=df["y"], random_state=42)

**Split 2 · Pool and dev.** `rest` is split again:

- **`dev`**: 200 messages, standing in for the **200 labels Priya can pay for**. We use their labels to check how good our labelling rules are.
- **`pool`**: everything else, standing in for the **unlabelled backlog**.

This split is stratified too, so both parts have the same share of spam.

In [3]:
# pool = the unlabelled backlog | dev = the 200 labels Priya can pay for | test = held back for scoring
pool, dev = train_test_split(rest, test_size=200, stratify=rest["y"], random_state=42)

**The sealed envelope.** In real life the backlog has no labels, so we hide the pool's true labels:

- `sealed_envelope` keeps a copy of the pool's labels, locked away until the end of Stage A. Then we open it to check how accurate our labelling functions were.
- `pool` is cut down to just the `text` column, so no later code can peek at the answers by accident.

The print confirms the three sizes.

In [4]:
sealed_envelope = pool["y"].copy()   # the pool's true labels: locked away until the end of Stage A
pool = pool[["text"]].copy()

print(f"pool: {len(pool):,} unlabelled | dev: {len(dev)} hand-labelled | test: {len(test)} for scoring")

pool: 1,828 unlabelled | dev: 200 hand-labelled | test: 800 for scoring


**Business assumptions.** The £ figures are estimates you can change later to see how the "best" model shifts:

- `COST_PER_LABEL`: what an annotator charges for one label.
- `COST_MISSED_SCAM` and `COST_BLOCKED_GENUINE`: what each kind of mistake costs. Blocking a genuine text is assumed to be **4× worse** than letting a scam through.
- `BACKLOG`: the 2 million unlabelled messages.

`SPAM`, `HAM` and `ABSTAIN` are the three votes a labelling function can return. `scoreboard` collects every model's results. The print shows why we can't just pay to label everything.

In [5]:
# Business assumptions for this ticket. They are estimates: change them later and see what happens.
COST_PER_LABEL = 0.08          # £ paid to an annotator for one label
COST_MISSED_SCAM = 5           # £ when a scam text reaches a customer (fraud losses, complaints)
COST_BLOCKED_GENUINE = 20      # £ when a genuine text is blocked (missed appointment, lost customer)
BACKLOG = 2_000_000            # messages waiting in the real backlog

SPAM, HAM, ABSTAIN = 1, 0, -1
scoreboard = []

print(f"Hand-labelling the whole backlog would cost £{BACKLOG * COST_PER_LABEL:,.0f}. "
      f"Priya's budget buys {len(dev)} labels (£{len(dev) * COST_PER_LABEL:.0f}).")

Hand-labelling the whole backlog would cost £160,000. Priya's budget buys 200 labels (£16).


**`train_model`** builds the classifier used all session:

1. **TF-IDF** turns each message into numbers: one column per word, weighted so words that are common in this message but rare overall count most.
2. **Logistic regression** learns which words push a message towards spam.

`**options` passes extra settings (such as `class_weight` in Stage C) straight to the logistic regression.

In [6]:
def train_model(texts, labels, **options):
    """TF-IDF + logistic regression, a standard baseline text classifier."""
    return make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000, **options)).fit(texts, labels)

**`evaluate`** scores a model on the 800 held-back `test` messages and adds one row to the scoreboard:

- **accuracy, precision, recall**: the usual metrics.
- **scams missed** (false negatives) and **genuine blocked** (false positives), from the confusion matrix.
- **cost £**: the mistakes priced with the business assumptions above.

If a model with the same name is evaluated again, its old row is replaced, so you can re-run cells safely.

In [7]:
def evaluate(name, model):
    """Score a model on the test set, cost its mistakes in pounds, and add it to the scoreboard."""
    pred = model.predict(test["text"])
    p, r, f1, _ = precision_recall_fscore_support(test["y"], pred, average="binary", zero_division=0)
    tn, fp, fn, tp = confusion_matrix(test["y"], pred, labels=[0, 1]).ravel()
    row = {"model": name, "accuracy": round(accuracy_score(test["y"], pred), 3), "precision": round(p, 2),
           "recall": round(r, 2), "scams missed": int(fn), "genuine blocked": int(fp),
           "cost £": int(fn * COST_MISSED_SCAM + fp * COST_BLOCKED_GENUINE)}
    scoreboard[:] = [s for s in scoreboard if s["model"] != name] + [row]
    return pd.DataFrame(scoreboard).set_index("model")

**`apply_lfs`** runs every labelling function on every message. The result is a table (matrix) with **one row per message and one column per LF**. Each cell holds `SPAM` (1), `HAM` (0) or `ABSTAIN` (-1).

In [8]:
def apply_lfs(lfs, texts):
    """One row per message, one column per labelling function."""
    return np.array([[lf(t) for lf in lfs] for t in texts])

**`lf_report`** grades each labelling function:

- **coverage**: share of messages it votes on (doesn't abstain).
- **conflict**: share of messages where it votes one way and another LF votes the opposite way.
- **accuracy on dev**: how often its votes match the 200 paid labels, counting only messages it voted on.

In [9]:
def lf_report(lfs, L, gold):
    """Coverage and conflict for each labelling function, and its accuracy on gold labels."""
    rows = []
    for j, lf in enumerate(lfs):
        fired = L[:, j] != ABSTAIN
        others = np.delete(L, j, axis=1)
        conflict = fired & ((others != ABSTAIN) & (others != L[:, [j]])).any(axis=1)
        rows.append({"LF": lf.__name__, "coverage": fired.mean(), "conflict": conflict.mean(),
                     "accuracy on dev": (L[fired, j] == gold[fired]).mean() if fired.any() else np.nan})
    return pd.DataFrame(rows).set_index("LF").round(2)

**`majority_vote`** combines all LF votes into one label per message: `SPAM` if more LFs said spam, `HAM` if more said ham, and `ABSTAIN` on a tie or when no LF voted. These combined labels are the cheap "weak labels" used to train a model in A3.

In [10]:
def majority_vote(L):
    """SPAM if more LFs voted spam, HAM if more voted ham, ABSTAIN on a tie or no votes."""
    spam, ham = (L == SPAM).sum(axis=1), (L == HAM).sum(axis=1)
    return np.where(spam > ham, SPAM, np.where(ham > spam, HAM, ABSTAIN))

---
## Stage A · Labelling (25 min)

> *"Before we automate anything, I want to know how hard this is for a person."* — Priya

### A1 · Be the annotator (5 min)

You are now the annotation team. **On your own, without talking**, label these six real messages: `1` = spam, `0` = ham.

In [11]:
snippets = ["Do you like Italian food", "You have won a £2,000 price", "has added you in their contact list",
            "order Interflora flowers", "JADE ITS PAUL", "As one of our registered subscribers"]
warmup = pd.concat([df[df["text"].str.contains(s, regex=False)].head(1) for s in snippets]).reset_index(drop=True)
for i, t in enumerate(warmup["text"]):
    print(f"[{i}] {t}\n")

[0] Do you like Italian food?

[1] 449050000301 You have won a £2,000 price! To claim, call 09050000301.

[2] Hi, Mobile no. <#> has added you in their contact list on www.fullonsms.com It s a great place to send free sms to people For more visit fullonsms.com

[3] INTERFLORA - It's not too late to order Interflora flowers for christmas call 0800 505060 to place your order before Midnight tomorrow.

[4] JADE ITS PAUL. Y DIDNT U TXT ME? DO U REMEMBER ME FROM BARMED? I WANT 2 TALK 2 U! TXT ME

[5] As one of our registered subscribers u can enter the draw 4 a 100 G.B. gift voucher by replying with ENTER. To unsubscribe text STOP



In [12]:
my_labels = [0, 1, 1, 1, 0, 1]          # ✍️ TODO: your six labels
neighbour_labels = [0, 1, 0, 0, 1, 1]   # ✍️ then swap and type in your neighbour's

me, them, gold = np.array(my_labels), np.array(neighbour_labels), warmup["y"].to_numpy()
print(f"You and your neighbour agree on {(me == them).sum()}/6")
print(f"You agree with the dataset on {(me == gold).sum()}/6, your neighbour on {(them == gold).sum()}/6")
print("Dataset labels:", gold.tolist())

You and your neighbour agree on 3/6
You agree with the dataset on 5/6, your neighbour on 4/6
Dataset labels: [0, 1, 0, 1, 0, 1]


**💬 Pair talk.** Where did you disagree? Is the flower-delivery advert a *scam*, or just *marketing* the customer may have signed up for? A real annotation team gets a written guideline for exactly this. Write the one sentence yours would need.

In [13]:
annotation_guideline = ("Label a message as spam (1) if it is unsolicited and tries to sell, promote, or get the recipient "
                        "to call, text, reply or click, even if the sender is a real company; personal messages between "
                        "people who know each other are ham (0).")

### A2 · Write labelling functions (12 min)

> *"The fraud analysts already know what scam texts look like. Can we capture what they know as code?"* — Priya

A **labelling function (LF)** encodes one rule of thumb. It reads a message and votes `SPAM`, votes `HAM`, or returns `ABSTAIN`. A fraud analyst has given you two.

In [14]:
def lf_contains_free(text):
    """Analyst: 'Scams often offer something free.'"""
    return SPAM if "free" in text.lower() else ABSTAIN


def lf_shouting(text):
    """Analyst: 'Messages mostly in CAPITALS are scams.'"""
    capitals = sum(ch.isupper() for ch in text)
    return SPAM if capitals / max(len(text), 1) > 0.3 else ABSTAIN

**✍️ TODO.** You are the analyst now. Write **three more** LFs. At least one must vote `HAM`.

Ideas: prize words (`win`, `claim`, `urgent`, `prize`), a `£` sign, a long run of digits (`re.search(r"\d{5,}", text)`), very short messages, personal words (`i'll`, `my`, `me`).

Run `pool.sample(10)` in a new cell to look at real messages first.

In [15]:
def lf_prize_words(text):
    """Prize words: 'You have WON a guaranteed prize, claim now' is a classic scam."""
    return SPAM if re.search(r"\b(win|won|winner|prize|claim|urgent|award|cash|guaranteed)\b", text.lower()) else ABSTAIN


def lf_long_number(text):
    """A long run of digits: premium-rate numbers (0905...) and short codes to text back."""
    return SPAM if re.search(r"\d{5,}", text) else ABSTAIN


def lf_short_plain(text):
    """Short and plain: under 60 characters with no £, long number or link is usually a friend."""
    return HAM if len(text) < 60 and not re.search(r"\d{4,}|£|http|www", text.lower()) else ABSTAIN


lfs = [lf_contains_free, lf_shouting, lf_prize_words, lf_long_number, lf_short_plain]

L_dev = apply_lfs(lfs, dev["text"])
assert (L_dev == HAM).any(), "None of your LFs votes HAM yet. Add one that does."
lf_report(lfs, L_dev, dev["y"].to_numpy())

,coverage,conflict,accuracy on dev
LF,,,
lf_contains_free,0.02,0.00,0.60
lf_shouting,0.02,0.01,0.00
lf_prize_words,0.06,0.00,1.00
lf_long_number,0.10,0.00,0.95
lf_short_plain,0.65,0.02,1.00


Reading the table: **coverage** is the share of messages an LF votes on, **conflict** is how often another LF votes the opposite way, and **accuracy on dev** checks its votes against the 200 paid labels.

**✍️ TODO.** One of the analyst's two rules is bad. Find it in the table, remove it from `lfs`, and improve any weak LF of your own. This is the real reason the 200 paid labels matter: they are how you catch an expert who is confidently wrong.

In [16]:
lfs = [lf for lf in lfs if lf is not lf_shouting]   # ✍️ name the bad LF

L_dev = apply_lfs(lfs, dev["text"])
weak_dev = majority_vote(L_dev)
voted = weak_dev != ABSTAIN
print(f"Combined by majority vote, your LFs label {voted.mean():.0%} of dev messages, "
      f"and {(weak_dev[voted] == dev['y'].to_numpy()[voted]).mean():.0%} of those labels are right.")
lf_report(lfs, L_dev, dev["y"].to_numpy())

Combined by majority vote, your LFs label 76% of dev messages, and 99% of those labels are right.


,coverage,conflict,accuracy on dev
LF,,,
lf_contains_free,0.02,0.0,0.60
lf_prize_words,0.06,0.0,1.00
lf_long_number,0.10,0.0,0.95
lf_short_plain,0.65,0.0,1.00


**Fixing the analyst's rule: `lf_shouting_2`.** The original `lf_shouting` was wrong on every dev message it voted on. Friends type in capitals too ("WHO ARE YOU SEEING?"), and short replies like "OK" are 100% capitals. The new version only votes `SPAM` when all three hold:

- at least **20 letters**, so short replies are ignored,
- more than **half of the letters** are capitals (digits and punctuation no longer count),
- it contains a **sales pitch**: a £ sign or words like *free*, *win*, *claim*, *reply*, *prize*, *ringtone*.

It replaces `lf_shouting` in the same slot of `lfs`, so all later cells use it.

In [17]:
def lf_shouting_2(text):
    """Analyst's rule, fixed: mostly CAPITALS *and* a sales pitch, in a message long enough to judge."""
    letters = [ch for ch in text if ch.isalpha()]
    caps_share = sum(ch.isupper() for ch in letters) / max(len(letters), 1)
    pitch = re.search(r"£|\b(free|win|won|claim|reply|prize|ringtone|tone)\b", text.lower())
    return SPAM if len(letters) >= 20 and caps_share > 0.5 and pitch else ABSTAIN


lfs = [lf_contains_free, lf_shouting_2, lf_prize_words, lf_long_number, lf_short_plain]

L_dev = apply_lfs(lfs, dev["text"])
weak_dev = majority_vote(L_dev)
voted = weak_dev != ABSTAIN
print(f"Combined by majority vote, your LFs label {voted.mean():.0%} of dev messages, "
      f"and {(weak_dev[voted] == dev['y'].to_numpy()[voted]).mean():.0%} of those labels are right.")
lf_report(lfs, L_dev, dev["y"].to_numpy())

Combined by majority vote, your LFs label 76% of dev messages, and 99% of those labels are right.


,coverage,conflict,accuracy on dev
LF,,,
lf_contains_free,0.02,0.0,0.60
lf_shouting_2,0.00,0.0,NaN
lf_prize_words,0.06,0.0,1.00
lf_long_number,0.10,0.0,0.95
lf_short_plain,0.65,0.0,1.00


### A3 · Train on your labels (8 min)

**🔮 Predict.** Model 1 trains on the **200 clean labels Priya paid for**. Model 2 trains on the **noisy labels your LFs give the whole pool**. Which misses fewer scams?

In [18]:
my_prediction = "model 2, because it leverages the weak labels from the LFs"   # ✍️ Model 1 or Model 2, and why

weak_pool = majority_vote(apply_lfs(lfs, pool["text"]))
labelled = weak_pool != ABSTAIN
share = labelled.mean()
print(f"Your LFs labelled {labelled.sum():,} of {len(pool):,} pool messages in under a second.")
print(f"At that rate they would label about {share * BACKLOG / 1e6:.1f} million of the backlog, "
      f"work that would cost £{share * BACKLOG * COST_PER_LABEL:,.0f} by hand.")

evaluate("A: 200 paid labels", train_model(dev["text"], dev["y"]))
evaluate("A: LF labels", train_model(pool["text"][labelled], weak_pool[labelled]))

Your LFs labelled 1,317 of 1,828 pool messages in under a second.


At that rate they would label about 1.4 million of the backlog, work that would cost £115,274 by hand.


,accuracy,precision,recall,scams missed,genuine blocked,cost £
model,,,,,,
A: 200 paid labels,0.891,0.0,0.0,87,0,435
A: LF labels,0.968,1.0,0.7,26,0,130


Now open the envelope: how good were your labels, and what would paying to label the whole pool have bought?

In [19]:
truth = sealed_envelope.to_numpy()
print(f"Your LF labels were right on {(weak_pool[labelled] == truth[labelled]).mean():.1%} of the messages they covered.")
evaluate("A: whole pool hand-labelled", train_model(pool["text"], truth))

Your LF labels were right on 98.2% of the messages they covered.


,accuracy,precision,recall,scams missed,genuine blocked,cost £
model,,,,,,
A: 200 paid labels,0.891,0.0,0.00,87,0,435
A: LF labels,0.968,1.0,0.70,26,0,130
A: whole pool hand-labelled,0.964,1.0,0.67,29,0,145


**💬 Pair talk.**

1. Look at Model 1: its **accuracy** next to its **scams missed**. Would you have shown Priya that accuracy figure? (Stage C returns to this.)
2. The lecture said *"more data isn't always better"*, yet here the bigger, noisier set won. Why?
3. Your LFs abstained on a large share of messages. In a real team, who should look at those?

---
## Stage B · Sampling (13 min)

> *"Compliance want to spot-check the backlog. They'll read 20 messages picked at random. Fine?"* — Priya

### B1 · The spot-check (6 min)

About 11% of messages are spam. **🔮 Predict:** in what percentage of random samples of 20 will the reviewer see **no spam at all**?

In [20]:
my_guess_percent = 10   ## (1- 0.11)^20

# ✍️ TODO: count the spam in a random sample of 20 messages. Hint: df.sample(n=..., random_state=seed)["y"].sum()
spam_counts = np.array([df.sample(n=20, random_state=seed)["y"].sum() for seed in range(2000)])

print(f"Your guess: {my_guess_percent}%   Simulated: {(spam_counts == 0).mean():.1%}   "
      f"Theory: {(1 - df['y'].mean()) ** 20:.1%}")

Your guess: 10%   Simulated: 11.0%   Theory: 10.1%


Roughly one spot-check in ten would report "no spam found" on a backlog that is 11% spam. That is the weakness of simple random sampling from the lecture: **no guarantee that rare classes are represented**. For fraud at 1%, four spot-checks in five would find nothing.

### B2 · Two analysts, one model (7 min)

> *"Sam and Aisha both tested last quarter's model and gave me different numbers. Who is wrong?"* — Priya

Possibly neither. **✍️ TODO.** Split the data 200 times with a plain random split and 200 times **stratified** by `y`, and compare how much the test set's spam rate moves.

In [21]:
random_rates, stratified_rates = [], []
for seed in range(200):
    _, te = train_test_split(df, test_size=100, random_state=seed)
    random_rates.append(te["y"].mean())
    _, te = train_test_split(df, test_size=100, random_state=seed, stratify=df["y"])   # ✍️
    stratified_rates.append(te["y"].mean())

print(f"Random splits:     spam rate from {min(random_rates):.0%} to {max(random_rates):.0%}")
print(f"Stratified splits: spam rate from {min(stratified_rates):.0%} to {max(stratified_rates):.0%}")

Random splits:     spam rate from 4% to 25%
Stratified splits: spam rate from 11% to 11%


**💬 Pair talk.** Explain to Priya in one sentence why Sam and Aisha disagreed, and what the team should do from now on.

---
## Stage C · Class imbalance (27 min)

> *"Bad news. Legal will only let us train on scams that were confirmed by the fraud team. That leaves us **40**."* — Priya

### C1 · The model that does nothing (6 min)

With 40 confirmed scams, spam is about 2% of the training data, the same situation as fraud detection or disease screening.

**🔮 Predict.** We train (1) a "model" that always says *ham* and (2) a real logistic regression. What accuracy will each get, and how many scams will the real one miss?

In [22]:
my_prediction = ("Both get about 89% accuracy, because 89% of test messages are ham. "
                 "The real model sees only 40 scams (~2%), so it will miss most of the 87 test scams.")   # ✍️

full = pd.concat([pool.assign(y=sealed_envelope), dev])
majority = full[full["y"] == 0]                                  # all the ham
minority = full[full["y"] == 1].sample(n=40, random_state=42)    # only 40 confirmed scams
train_imb = pd.concat([majority, minority])
print(f"Training data: {len(majority):,} ham, {len(minority)} spam ({train_imb['y'].mean():.1%} spam)")

evaluate("C: always predict ham", DummyClassifier(strategy="most_frequent").fit(train_imb["text"], train_imb["y"]))
evaluate("C: baseline", train_model(train_imb["text"], train_imb["y"])).tail(2)

Training data: 1,809 ham, 40 spam (2.2% spam)


,accuracy,precision,recall,scams missed,genuine blocked,cost £
model,,,,,,
C: always predict ham,0.891,0.0,0.0,87,0,435
C: baseline,0.891,0.0,0.0,87,0,435


A trained model that behaves like one that does nothing, with an accuracy you could put on a slide. Accuracy is a **symmetric** metric, so from here on read **recall, precision and the cost column**: what each model's mistakes on 800 test messages would cost the business.

### C2 · Data-level fix: resampling (7 min)

**✍️ TODO.** Build two balanced training sets with `resample`:

- **undersample**: cut the ham down to the number of spam messages (`replace=False`)
- **oversample**: repeat the spam up to the number of ham messages (`replace=True`)

**🔮 Predict:** which one will be more expensive, and why?

In [23]:
# resample(frame, replace=True or False, n_samples=..., random_state=42)
under = pd.concat([resample(majority, replace=False, n_samples=len(minority), random_state=42), minority])
over = pd.concat([majority, resample(minority, replace=True, n_samples=len(majority), random_state=42)])

assert under["y"].mean() == 0.5 and len(under) == 2 * len(minority), "under should be 40 ham + 40 spam"
assert over["y"].mean() == 0.5 and len(over) == 2 * len(majority), "over should have as much spam as ham"

evaluate("C: undersampled", train_model(under["text"], under["y"]))
evaluate("C: oversampled", train_model(over["text"], over["y"])).tail(3)

,accuracy,precision,recall,scams missed,genuine blocked,cost £
model,,,,,,
C: baseline,0.891,0.00,0.00,87,0,435
C: undersampled,0.917,0.57,0.93,6,60,1230
C: oversampled,0.969,1.00,0.71,25,0,125


**💬 Pair talk.** One of these catches the most scams *and* costs the most. How can both be true? And why did we resample the training data only, never the test set?

### C3 · Algorithm-level fix, and a challenge (8 min)

Instead of changing the data, change the **loss**, so a missed scam costs the model more than a missed ham message. In scikit-learn that is the `class_weight` argument.

**✍️ TODO.** Train on the original `train_imb` with `class_weight="balanced"`, then with a weight of your own such as `{0: 1, 1: 50}`.

**🏆 Challenge:** find the spam weight that gives the **lowest cost £**. Lowest in the room wins.

In [24]:
balanced = train_model(train_imb["text"], train_imb["y"], class_weight="balanced")
my_weight = 100   # ✍️ swept 1–200: cost falls until ~100 (£65), then blocked genuine texts push it back up
tuned = train_model(train_imb["text"], train_imb["y"], class_weight={0: 1, 1: my_weight})

evaluate("C: class_weight balanced", balanced)
evaluate("C: class_weight tuned", tuned)

,accuracy,precision,recall,scams missed,genuine blocked,cost £
model,,,,,,
A: 200 paid labels,0.891,0.00,0.00,87,0,435
A: LF labels,0.968,1.00,0.70,26,0,130
A: whole pool hand-labelled,0.964,1.00,0.67,29,0,145
C: always predict ham,0.891,0.00,0.00,87,0,435
C: baseline,0.891,0.00,0.00,87,0,435
C: undersampled,0.917,0.57,0.93,6,60,1230
C: oversampled,0.969,1.00,0.71,25,0,125
C: class_weight balanced,0.976,1.00,0.78,19,0,95
C: class_weight tuned,0.984,1.00,0.85,13,0,65


### C4 · Will it catch today's scams? (4 min)

The training messages are from the 2000s. Here are texts of the kind people report now. Run your tuned model on them, then **✍️ add two of your own**: one scam you have received and one genuine message.

In [25]:
todays_texts = [
    "Your parcel could not be delivered. Pay the £1.99 redelivery fee at parcel-redelivery.example.com",
    "Hi Mum, I dropped my phone and this is my new number. Can you send me £200 for a bill today? I will explain later",
    "Your bank account has been suspended. Verify your identity at secure-login.example.com to avoid closure",
    "You are owed a tax refund of £276.40. Claim it now at tax-refund-claims.example.com",
    "Your verification code is 482913. Do not share it with anyone.",
    "Running 10 mins late, see you at the station",
    # ✍️ add yours here, each in quotes and followed by a comma
    "Royal Mail: your parcel is on hold due to unpaid customs. Pay £2.99 within 24 hours at royalmail-customs.example.com",
    "Can you pick up some milk on your way home? Thanks x",
]
pd.DataFrame({"text": todays_texts,
              "model says": np.where(tuned.predict(todays_texts) == SPAM, "SCAM", "ok"),
              "P(scam)": tuned.predict_proba(todays_texts)[:, 1].round(2)})

,text,model says,P(scam)
0,Your parcel could not be delivered. Pay the £1.99 redelivery fee at parcel-redelivery.example.com,ok,0.15
1,"Hi Mum, I dropped my phone and this is my new number. Can you send me £200 for a bill today? I will explain later",ok,0.05
2,Your bank account has been suspended. Verify your identity at secure-login.example.com to avoid closure,SCAM,0.58
3,You are owed a tax refund of £276.40. Claim it now at tax-refund-claims.example.com,SCAM,0.62
4,Your verification code is 482913. Do not share it with anyone.,ok,0.18
5,"Running 10 mins late, see you at the station",ok,0.06
6,Royal Mail: your parcel is on hold due to unpaid customs. Pay £2.99 within 24 hours at royalmail-customs.example.com,ok,0.15
7,Can you pick up some milk on your way home? Thanks x,ok,0.05


**💬 Pair talk.** Which scams got through, and why can a model trained on prize draws and ringtones not see them? This is **data drift**: the world moved on after the training data was collected. What would you ask Priya for?

---
## Wrap-up (5 min)

### Reply to the ticket

Priya needs three sentences on Friday. Write them: which model you would ship, what its mistakes would cost, and one risk she should know about.

In [26]:
reply_to_priya = (   # ✍️ TODO: three sentences
    "We should ship the TF-IDF + logistic regression filter trained with a spam class weight of 100, which on 800 test "
    "messages caught 85% of scams (13 missed) and blocked no genuine texts, the lowest cost of every model we tried.\n"
    "Its mistakes would cost about £65 per 800 messages, roughly 8p a message or about £160,000 across the 2 million "
    "backlog, compared with £435 per 800 if we had no filter at all.\n"
    "The main risk is data drift: it was trained on old-style scams and missed 3 of 5 modern ones (parcel fees and "
    "'Hi Mum' texts), so we need recent 7726 reports labelled and the model retrained regularly."
)
print(reply_to_priya)
pd.DataFrame(scoreboard).set_index("model").sort_values("cost £")

We should ship the TF-IDF + logistic regression filter trained with a spam class weight of 100, which on 800 test messages caught 85% of scams (13 missed) and blocked no genuine texts, the lowest cost of every model we tried.
Its mistakes would cost about £65 per 800 messages, roughly 8p a message or about £160,000 across the 2 million backlog, compared with £435 per 800 if we had no filter at all.
The main risk is data drift: it was trained on old-style scams and missed 3 of 5 modern ones (parcel fees and 'Hi Mum' texts), so we need recent 7726 reports labelled and the model retrained regularly.


,accuracy,precision,recall,scams missed,genuine blocked,cost £
model,,,,,,
C: class_weight tuned,0.984,1.00,0.85,13,0,65
C: class_weight balanced,0.976,1.00,0.78,19,0,95
C: oversampled,0.969,1.00,0.71,25,0,125
A: LF labels,0.968,1.00,0.70,26,0,130
A: whole pool hand-labelled,0.964,1.00,0.67,29,0,145
A: 200 paid labels,0.891,0.00,0.00,87,0,435
C: always predict ham,0.891,0.00,0.00,87,0,435
C: baseline,0.891,0.00,0.00,87,0,435
C: undersampled,0.917,0.57,0.93,6,60,1230


### Commit your work

```bash
cd ..
pip freeze > requirements.txt
git add .
git commit -m "Week 03: labelling, sampling, class imbalance"
git push
```

Then stop your Codespace (Command Palette → *Codespaces: Stop Current Codespace*).

**Exit ticket.** In one word, what is something you learned today?